In [15]:
!pip install PIL
!pip install scipy
import numpy as np
import tkinter as tk
from tkinter import *
from tkinter import filedialog
from PIL import Image, ImageTk
from scipy.signal import convolve2d


canvas = None
image_tk = None
matrice_pixels = None
nom_fichier = None
PhotoImage = None
historique=[]
indice_historique = -1

def charger_fichier(element):
    global img
    global img_tk
    global canvas 
    global matrice_pixels
    nom_fichier = str(filedialog.askopenfilename(title="Ouvrir une image", filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp")])) #ouvre l'explorateur
    if nom_fichier != "":
        img=Image.open(nom_fichier) #ouvre l'image et applique le filtre vert
        redim=img.resize((600,500))
        img_tk= ImageTk.PhotoImage(redim)#convertit l'image en format affichable par Tkinter
        # if canvas is None: ##on créer un nouveau canvas s'il n'existe pas
        #     canvas=tk.Canvas(element, width= img.size[0], height=img.size[1])
        #     canvas.pack()
        
        # else: ##image déjà créer
        #     canvas.delete("all")
        #     canvas.config(width= img.size[0], height=img.size[1])
        #     element.update_idletasks()
        #toutes ces lignes ne servent à rien car elles sont manipubalble pour une seul filtre
        canvas.delete("all")
        canvas.config(width= 600, height=500) #sert à forcer  l'ajustement de la taille de l'image en fonction du canvas
        canvas.create_image(0,0, anchor=tk.NW, image=img_tk) #affichage de l'image 
        element.update_idletasks() #force la mise à jour de l'élément pour charger un nouveau élément si besoin en cliquant sur Ouvrir à chaque fois
        matrice_pixels = np.array(img)
        historique_image()

def historique_image():
    global historique, indice_historique, matrice_pixels
    #matrice_pixels = np.array(img)
    if len(historique)-1 > indice_historique: #si on annuler plusieur fois on  ne veut pas garder les images dans la liste après l'annulation
        historique=historique[:indice_historique+1] #on créer une nouvelle liste en partant du début et s'arretant à l'image actuelle (attention pas oublier que +1 c'est pour le décalage de indice_hostorique !!))
    historique.append(matrice_pixels.copy()) #mise à jour de la liste à chaque nouveau chargement de fichier
    indice_historique= len(historique)-1 #but : afficher la dernière image générée/affichée -> on prend l'avant dernier indice de la liste historique pour renvoyer l'image correspondante
    
def annuler_filtre():
    global indice_historique, historique, matrice_pixels, img
    if indice_historique > 0:
        indice_historique -= 1
        matrice_pixels=historique[indice_historique].copy()
        img = Image.fromarray(matrice_pixels.astype(np.uint8))
        rafraichir()

def retablir_filtre():
    global indice_historique, matrice_pixels, historique, img
    if len(historique)-1 > indice_historique:
        indice_historique += 1
        matrice_pixels= historique[indice_historique].copy()
        print("pixel[0,0] après rétablir:", matrice_pixels[0,0]) 
        img = Image.fromarray(matrice_pixels.astype(np.uint8))
        rafraichir()


def rafraichir():
    global photo, img, img_tk, canvas, matrice_pixels
    img=Image.fromarray(matrice_pixels.astype(np.uint8))
    image_redim = img.resize((600, 500))
    photo= ImageTk.PhotoImage(image_redim)
    canvas.delete("all")
    canvas.config(width=photo.width(), height=photo.height())
    canvas.create_image(0, 0, anchor=tk.NW, image=photo)
    canvas.image=photo #pour que tkinter continue d'afficher la photo/ image
    fenetre_principale.update_idletasks()

def filtre_vert(img):
    global matrice_pixels
    matrice_pixels=np.array(img)
    matrice_pixels[:,:,[0,2]]=0 #supprimer composantes R[0] et B[2] (par défaut)
    rafraichir()
    return Image.fromarray(matrice_pixels.astype(np.uint8)) #format de conversion de numpy pour passer da la matrice à l'image

def sepia(img):
    global matrice_pixels
    matrice_pixels = np.array(img)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= r+0.5*v+0.1*b
    v_prime= 0.5*r+0.8*v+0.1*b
    b_prime= 0.2*r+0.3*v+0.5*b
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    matrice_pixels[:,:,[0]]=r_prime
    matrice_pixels[:,:,[1]]=v_prime
    matrice_pixels[:,:,[2]]=b_prime
    rafraichir()
    return Image.fromarray(matrice_pixels.astype(np.uint8))

def miroir(img):
    global matrice_pixels
    matrice_pixels = np.array(img)
    matrice_pixels= matrice_pixels[:,::-1,:] #inverse les colonnes avec le slicing
    rafraichir()
    return Image.fromarray(matrice_pixels.astype(np.uint8))

def luminosite(img):
    global matrice_pixels
    matrice_pixels = np.array(img)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= r+50
    v_prime= v+50
    b_prime= b+50
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    matrice_pixels[:,:,[0]]=r_prime
    matrice_pixels[:,:,[1]]=v_prime
    matrice_pixels[:,:,[2]]=b_prime
    rafraichir()
    return Image.fromarray(matrice_pixels.astype(np.uint8))

def noir_et_blanc(img):
    global matrice_pixels
    matrice_pixels = np.array(img)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= 0.299*r
    v_prime= 0.587*v
    b_prime= 0.114*b
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    Y= r_prime+v_prime+b_prime
    matrice_pixels[:,:,[0,1,2]]=Y
    rafraichir()
    return Image.fromarray(matrice_pixels.astype(np.uint8))

#Filtrage gaussien

#def convolve importée du TD pour comprendre le principe de la moyenne et de la fenetre glissante et le padding pour garder les valeurs de pixels dans l'image et ne pas dépasser : fonction de np => np.pad
# def convolve(img, kernel):
#     """
#     Applique une convolution 2D sur une image avec un noyau donné.
    
#     :param image: Image 2D (grayscale) ou un canal d'image (array NumPy).
#     :param kernel: Noyau de convolution (array NumPy 2D).
#     :return: Image convoluée (array NumPy 2D).
#     """
#     # Dimensions de l'image et du noyau
#     image_height, image_width = image.shape
#     kernel_height, kernel_width = kernel.shape

#     # Calcul des marges pour le padding
#     pad_h = kernel_height // 2
#     pad_w = kernel_width // 2

#     # Ajouter un padding autour de l'image
#     padded_image = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')

#     # Créer une image de sortie vide
#     output = np.zeros_like(image)

#     # Appliquer la convolution
#     for i in range(image_height):
#         for j in range(image_width):
#             # Extraire la région locale autour du pixel
#             region = padded_image[i:i + kernel_height, j:j + kernel_width]
#             # Calculer le produit élément par élément et la somme
#             output[i, j] = np.sum(region * kernel)
    
#     return output

# noyau = np.array([
#     [1, 2, 1],
#     [2, 4, 2],
#     [1, 2, 1]
#     ]) / 16
#=> trop petit pour voir une diff sur une image en 600x500 donc on prend une matrice plus grande

noyau = np.array([
    [1,  4,  6,  4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4],
    [1,  4,  6,  4, 1]
    ]) / 256
# Algorithmes de filtre

def flou_Gaussien():
    global matrice_pixels,noyau,img
    #matrice du même type en float pour ne pas avoir d'arrondis et de la même taille que matrice_pixels
    float_matrice=matrice_pixels.astype(float)
    new_matrice = np.zeros_like(float_matrice)
    for i  in range (3): #convolution sur les 3 canaux; boundary = symm => évite bords noirs
        new_matrice[:,:,i] = convolve2d(float_matrice[:,:,i],noyau, mode="same",boundary="symm")
    matrice_pixels=np.clip(new_matrice, 0, 255).astype(np.uint8)
    img=Image.fromarray(matrice_pixels.astype(np.uint8)) #si on charge un nouveau filtre après, l'image sera MAJ et on appliquera pas le deuxieme filtre sur le flou
    rafraichir()
    
    



def callback_afficher():
    return charger_fichier(fenetre_principale)

def callback_filtre_vert():
    global img, matrice_pixels
    filtre_vert(img)
    img = Image.fromarray(matrice_pixels.astype(np.uint8))
    historique_image() #a la fin pour suvegarder l'etat de l'image en matrice et pour que l'indice_historique marche bien en fonction de annuler ou retablir
    

def callback_sepia():
    global img, matrice_pixels
    sepia(img)
    img = Image.fromarray(matrice_pixels.astype(np.uint8))
    historique_image()
    

def callback_miroir():
    global img, matrice_pixels
    miroir(img)
    img = Image.fromarray(matrice_pixels.astype(np.uint8))
    historique_image()


def callback_luminosite():
    global img, matrice_pixels
    luminosite(img)
    img = Image.fromarray(matrice_pixels.astype(np.uint8))
    historique_image()

def callback_noiretblanc():
    global img, matrice_pixels
    noir_et_blanc(img)
    img = Image.fromarray(matrice_pixels.astype(np.uint8))
    historique_image()

def callback_gauss():
    flou_Gaussien()
    historique_image()




fenetre_principale=tk.Tk()
fenetre_principale.title("Application de Filtres")
fenetre_principale.geometry('800x700')
fenetre_principale['bg']="lightpink"
fenetre_principale.resizable(True, True) #la fenetre ne peut plus etre redimensionnée par l'utilisateur, on a définit une taille par défaut


#espace pour dessiner le composant graphique, ici l'espace pour afficher l'image et y appliquer les filtres
width, height = 600, 500
canvas=Canvas(fenetre_principale, width=width, height=height, bg='white', bd=0, highlightthickness=0)
canvas.pack(padx=10, pady=10)
frame_menu=tk.Frame()
frame_menu.pack()

#les 2 onglets de l'appli : Fichier et Effets

barre_menu = tk.Menu(frame_menu, bg="#F4EDE0", tearoff=0) #le tearoff sert à ne pas détacher l'onglet du menu de la fenetre principale
menu_fichier= tk.Menu(barre_menu, tearoff=0)# contient "ouvrir" & lié au callback_ouvrir : charge l'image depuis le disque dur, la stocke dans matrice pixel
# menu_fichier & menu_effets sont des sous onglets (on va soit y mettre directement l'action qu'on souhaite exécuter soit on y met un menu déroulant)

menu_fichier.add_command(label="Ouvrir", command=callback_afficher)
menu_fichier.add_command(label="Annuler le filtre", command=annuler_filtre)
menu_fichier.add_command(label="Rétablir le filtre", command=retablir_filtre)
barre_menu.add_cascade(label=" Fichier", menu=menu_fichier) #on affiche le menu deroulant
fenetre_principale.config(menu=barre_menu)


menu_effets=tk.Menu(frame_menu, tearoff=0)
menu_effets.add_command(label= "filtre vert", command=callback_filtre_vert)
menu_effets.add_command(label="filtre sépia", command=callback_sepia)
menu_effets.add_command(label="filtre miroir", command=callback_miroir)
menu_effets.add_command(label="filtre luminosite", command=callback_luminosite)
menu_effets.add_command(label="filtre noir et blanc", command=callback_noiretblanc)
menu_effets.add_command(label="filtre Gaussien", command=callback_gauss)
barre_menu.add_cascade(label="Effets", menu=menu_effets) #on affiche le menu deroulant
fenetre_principale.config(menu=barre_menu)



fenetre_principale.mainloop()



ERROR: Could not find a version that satisfies the requirement PIL (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for PIL



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
